In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -U typing_extensions

import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # Only for Gemma 3N

In [ ]:
!pip install opencv-python

In [ ]:
!pip uninstall torchcodec -y

In [ ]:
from huggingface_hub import login
import os

# Paste your token here (get it from: https://huggingface.co/settings/tokens)
HF_TOKEN = "" 

login(token=HF_TOKEN)

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

# Load the 2B finetuned model using transformers directly
processor = AutoProcessor.from_pretrained("blind-assist/gemma-3n-2b-finetune-e1-8500")
model = AutoModelForImageTextToText.from_pretrained(
    "blind-assist/gemma-3n-2b-finetune-e1-8500",
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

print(f"Model loaded on: {model.device}")

In [ ]:
import cv2
import numpy as np
from PIL import Image

def downsample_video(video_path, num_frames=2):
    """Extracts evenly spaced frames for VLM context."""
    
    vidcap = cv2.VideoCapture(video_path)
    total_frames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0: return []
    
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    for i in indices:
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, i)
        success, image = vidcap.read()
        if success:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(image))
    vidcap.release()
    return frames

In [ ]:
import os
import time
import csv
from pathlib import Path

# Setup paths
video_folder = "videos"
output_csv = "video_guidance_results_2b.csv"

# Get all video files
video_files = sorted([f for f in os.listdir(video_folder) 
                     if f.endswith(('.mp4', '.avi', '.mov', '.mkv'))])

print(f"Found {len(video_files)} videos to process")

# Instruction for the model
instruction = ("Given the visual input from the user's forward perspective, "
               "generate exactly one short sentence to guide a visually impaired user "
               "by identifying critical obstacles or landmarks, describing their locations "
               "using clock directions relative to the user (12 o'clock is straight ahead), "
               "including relevant details such as size, material, or distance, and giving "
               "one clear action, while prioritizing immediate safety and avoiding any extra explanation.")

# Prepare CSV file
results = []

# Process each video
for idx, video_file in enumerate(video_files, 1):
    video_path = os.path.join(video_folder, video_file)
    print(f"\n[{idx}/{len(video_files)}] Processing: {video_file}")
    
    try:
        # Extract frames from video
        frames = downsample_video(video_path)
        
        if not frames:
            raise ValueError("No frames extracted from video")
        
        # Build the message with images and text
        content = []
        for img in frames:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": instruction})
        
        messages = [{"role": "user", "content": content}]
        
        # Process inputs using apply_chat_template (this works for the finetuned model)
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        start_time = time.time()

        # Generate output
        print(f"Generating guidance for {video_file}...")
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=True,
            temperature=1.0,
            top_p=0.95,
            top_k=64
        )

        end_time = time.time()
        inference_time = round(end_time - start_time, 3)
        
        # Decode only the generated tokens (excluding the input)
        generated_text = processor.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        
        # Clean up the output
        guidance = " ".join(generated_text.strip().split())
        
        print(f"Output: {guidance}")
        
        # Store result
        results.append({
            "video_id": idx,
            "video_filename": video_file,
            "guidance": guidance,
            "inference_time_s": inference_time
        })
        
    except Exception as e:
        import traceback
        print(f"Error processing {video_file}: {str(e)}")
        traceback.print_exc()
        results.append({
            "video_id": idx,
            "video_filename": video_file,
            "guidance": f"ERROR: {str(e)}",
            "inference_time_s": 0.0
        })

# Save results to CSV
print(f"\nSaving results to {output_csv}...")
with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['video_id', 'video_filename', 'guidance', 'inference_time_s']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    writer.writerows(results)

print(f"\nProcessing complete! Results saved to {output_csv}")
print(f"Successfully processed {len([r for r in results if not r['guidance'].startswith('ERROR')])} out of {len(video_files)} videos")

In [ ]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")